In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "5d733eab198ad58f23ceee3f1550014385366ece"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/5d733eab198ad58f23ceee3f1550014385366ece/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Single Shot Multibox Detection

In that section to that section,
we introduced bounding boxes, anchor boxes,
multiscale object detection, and the dataset for object detection.
Now we are ready to use such background
knowledge to design an object detection model:
single shot multibox detection
(SSD) [@Liu.Anguelov.Erhan.ea.2016].
This model is simple, fast, and widely used.
Although this is just one of vast amounts of
object detection models,
some of the design principles
and implementation details in this section
are also applicable to other models.


## Model

the figure provides an overview of
the design of single-shot multibox detection.
This model mainly consists of
a base network
followed by
several multiscale feature map blocks.
The base network
is for extracting features from the input image,
so it can use a deep CNN.
For example,
the original single-shot multibox detection paper
adopts a VGG network truncated before the
classification layer [@Liu.Anguelov.Erhan.ea.2016],
while ResNet has also been commonly used.
Through our design
we can make the base network output
larger feature maps
so as to generate more anchor boxes
for detecting smaller objects.
Subsequently,
each multiscale feature map block
reduces (e.g., by half)
the height and width of the feature maps
from the previous block,
and enables each unit
of the feature maps
to increase its receptive field on the input image.


Recall the design
of multiscale object detection
through layerwise representations of images by
deep neural networks
in that section.
Since
multiscale feature maps closer to the top of the figure
are smaller but have larger receptive fields,
they are suitable for detecting
fewer but larger objects.

In a nutshell,
via its base network and several multiscale feature map blocks,
single-shot multibox detection
generates a varying number of anchor boxes with different sizes,
and detects varying-size objects
by predicting classes and offsets
of these anchor boxes (thus the bounding boxes);
this is therefore a multiscale object detection model.


![As a multiscale object detection model, single-shot multibox detection mainly consists of a base network followed by several multiscale feature map blocks.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/ssd.svg)


In the following,
we will describe the implementation details
of different blocks in the figure. To begin with, we discuss how to implement
the class and bounding box prediction.


### Class Prediction Layer

Let the number of object classes be $q$.
Then anchor boxes have $q+1$ classes,
where class 0 is background.
At some scale,
suppose that the height and width of feature maps
are $h$ and $w$, respectively.
When $a$ anchor boxes
are generated with
each spatial position of these feature maps as their center,
a total of $hwa$ anchor boxes need to be classified.
This often makes classification with fully connected layers infeasible due to likely
heavy parametrization costs.
Recall how we used channels of
convolutional layers
to predict classes in that section.
Single-shot multibox detection uses the
same technique to reduce model complexity.

Specifically,
the class prediction layer uses a convolutional layer
without altering width or height of feature maps.
In this way,
there can be a one-to-one correspondence
between outputs and inputs
at the same spatial dimensions (width and height)
of feature maps.
More concretely,
channels of the output feature maps
at any spatial position ($x$, $y$)
represent class predictions
for all the anchor boxes centered on
($x$, $y$) of the input feature maps.
To produce valid predictions,
there must be $a(q+1)$ output channels,
where for the same spatial position
the output channel with index $i(q+1) + j$
represents the prediction of
the class $j$ ($0 \leq j \leq q$)
for the anchor box $i$ ($0 \leq i < a$).

Below we define such a class prediction layer,
specifying $a$ and $q$ via arguments `num_anchors` and `num_classes`, respectively.
This layer uses a $3\times3$ convolutional layer with a
padding of 1.
The width and height of the input and output of this
convolutional layer remain unchanged.

In [ ]:
%matplotlib inline
from d2l import tensorflow as d2l
import tensorflow as tf
from tensorflow import keras
import numpy as np

def cls_predictor(num_anchors, num_classes):
    return keras.layers.Conv2D(num_anchors * (num_classes + 1),
                               kernel_size=3, padding='same')

### Bounding Box Prediction Layer

The design of the bounding box prediction layer is similar to that of the class prediction layer.
The only difference lies in the number of outputs for each anchor box:
here we need to predict four offsets rather than $q+1$ classes.

In [ ]:
def bbox_predictor(num_anchors):
    return keras.layers.Conv2D(num_anchors * 4, kernel_size=3, padding='same')

### Concatenating Predictions for Multiple Scales

As we mentioned, single-shot multibox detection
uses multiscale feature maps to generate anchor boxes and predict their classes and offsets.
At different scales,
the shapes of feature maps
or the numbers of anchor boxes centered on the same unit
may vary.
Therefore,
shapes of the prediction outputs
at different scales may vary.

In the following example,
we construct feature maps at two different scales,
`Y1` and `Y2`,
for the same minibatch,
where the height and width of `Y2`
are half of those of `Y1`.
Let's take class prediction as an example.
Suppose that
5 and 3 anchor boxes
are generated for every unit in `Y1` and `Y2`, respectively.
Suppose further that
the number of object classes is 10.
For feature maps `Y1` and `Y2`
the numbers of channels in the class prediction outputs
are $5\times(10+1)=55$ and $3\times(10+1)=33$, respectively,
where either output shape is
(batch size, number of channels, height, width).

In [ ]:
def forward(x, block):
    # Keras uses NHWC format; input shape: (N, H, W, C)
    return block(x)

Y1 = forward(tf.zeros((2, 20, 20, 8)), cls_predictor(5, 10))
Y2 = forward(tf.zeros((2, 10, 10, 16)), cls_predictor(3, 10))
Y1.shape, Y2.shape

As we can see, except for the batch size dimension,
the other three dimensions all have different sizes.
To concatenate these two prediction outputs for more efficient computation,
we will transform these tensors into a more consistent format.

Note that
the channel dimension holds the predictions for
anchor boxes with the same center.
We first move this dimension to the innermost.
Since the batch size remains the same for different scales,
we can transform the prediction output
into a two-dimensional tensor
with shape (batch size, height $\times$ width $\times$ number of channels).
Then we can concatenate
such outputs at different scales
along dimension 1.

In [ ]:
def flatten_pred(pred):
    # pred is (N, H, W, C); flatten H*W*C with channel innermost so that
    # the resulting layout matches multibox_prior's anchor ordering
    return tf.reshape(pred, (tf.shape(pred)[0], -1))

def concat_preds(preds):
    return tf.concat([flatten_pred(p) for p in preds], axis=1)

In this way,
even though `Y1` and `Y2` have different sizes
in channels, heights, and widths,
we can still concatenate these two prediction outputs at two different scales for the same minibatch.

In [ ]:
concat_preds([Y1, Y2]).shape

### Downsampling Block

In order to detect objects at multiple scales,
we define the following downsampling block `down_sample_blk` that
halves the height and width of input feature maps.
In fact,
this block applies the design of VGG blocks
in that section.
More concretely,
each downsampling block consists of
two $3\times3$ convolutional layers with padding of 1
followed by a $2\times2$ max-pooling layer with stride of 2.
As we know, $3\times3$ convolutional layers with padding of 1 do not change the shape of feature maps.
However, the subsequent $2\times2$ max-pooling  reduces the height and width of input feature maps by half.
For both input and output feature maps of this downsampling block,
because $1\times 2+(3-1)+(3-1)=6$,
each unit in the output
has a $6\times6$ receptive field on the input.
Therefore, the downsampling block enlarges the receptive field of each unit in its output feature maps.

In [ ]:
def down_sample_blk(num_channels):
    blk = keras.Sequential()
    for _ in range(2):
        blk.add(keras.layers.Conv2D(num_channels, kernel_size=3,
                                    padding='same'))
        blk.add(keras.layers.BatchNormalization())
        blk.add(keras.layers.ReLU())
    blk.add(keras.layers.MaxPool2D(pool_size=2, strides=2))
    return blk

In the following example, our constructed downsampling block changes the number of input channels and halves the height and width of the input feature maps.

In [ ]:
forward(tf.zeros((2, 20, 20, 3)), down_sample_blk(10)).shape

### Base Network Block

The base network block is used to extract features from input images.
For simplicity,
we construct a small base network
consisting of three downsampling blocks
that double the number of channels at each block.
Given a $256\times256$ input image,
this base network block outputs $32 \times 32$ feature maps ($256/2^3=32$).

In [ ]:
def base_net():
    blk = keras.Sequential()
    for num_filters in [16, 32, 64]:
        blk.add(down_sample_blk(num_filters))
    return blk

forward(tf.zeros((2, 256, 256, 3)), base_net()).shape

### The Complete Model


The complete
single shot multibox detection model
consists of five blocks.
The feature maps produced by each block
are used for both
(i) generating anchor boxes
and (ii) predicting classes and offsets of these anchor boxes.
Among these five blocks,
the first one
is the base network block,
the second to the fourth are
downsampling blocks,
and the last block
uses global max-pooling
to reduce both the height and width to 1.
Technically,
the second to the fifth blocks
are all
those
multiscale feature map blocks
in the figure.

In [ ]:
def get_blk(i):
    if i == 0:
        return base_net()
    elif i == 4:
        return keras.layers.GlobalMaxPool2D(keepdims=True)
    else:
        return down_sample_blk(128)

Now we define the forward propagation
for each block.
Different from
in image classification tasks,
outputs here include
(i) CNN feature maps `Y`,
(ii) anchor boxes generated using `Y` at the current scale,
and (iii) classes and offsets predicted (based on `Y`)
for these anchor boxes.

In [ ]:
def blk_forward(X, blk, size, ratio, cls_predictor, bbox_predictor,
                training=False):
    Y = blk(X, training=training)
    # Keras uses NHWC; multibox_prior expects NCHW (shape[-2:] = H, W)
    Y_nchw = tf.transpose(Y, (0, 3, 1, 2))
    anchors = d2l.multibox_prior(Y_nchw, sizes=size, ratios=ratio)
    # Keep cls_preds and bbox_preds in NHWC; flatten_pred relies on the
    # channel-last memory layout to align with multibox_prior's anchor order
    cls_preds = cls_predictor(Y, training=training)
    bbox_preds = bbox_predictor(Y, training=training)
    return (Y, anchors, cls_preds, bbox_preds)

Recall that
in the figure
a multiscale feature map block
that is closer to the top
is for detecting larger objects;
thus, it needs to generate larger anchor boxes.
In the above forward propagation,
at each multiscale feature map block
we pass in a list of two scale values
via the `sizes` argument
of the invoked `multibox_prior` function (described in that section).
In the following,
the interval between 0.2 and 1.05
is split evenly
into five sections to determine the
smaller scale values at the five blocks: 0.2, 0.37, 0.54, 0.71, and 0.88.
Then their larger scale values
are given by
$\sqrt{0.2 \times 0.37} = 0.272$, $\sqrt{0.37 \times 0.54} = 0.447$, and so on.

In [ ]:
sizes = [[0.2, 0.272], [0.37, 0.447], [0.54, 0.619], [0.71, 0.79],
         [0.88, 0.961]]
ratios = [[1, 2, 0.5]] * 5
num_anchors = len(sizes[0]) + len(ratios[0]) - 1

Now we can define the complete model `TinySSD` as follows.

In [ ]:

class TinySSD(keras.Model):
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.cls_loss = keras.losses.SparseCategoricalCrossentropy(
            from_logits=True)
        for i in range(5):
            # Equivalent to: self.blk_i, self.cls_i, self.bbox_i
            setattr(self, f'blk_{i}', get_blk(i))
            setattr(self, f'cls_{i}', cls_predictor(num_anchors, num_classes))
            setattr(self, f'bbox_{i}', bbox_predictor(num_anchors))

    def call(self, X, training=False):
        anchors_list = [None] * 5
        cls_preds_list = [None] * 5
        bbox_preds_list = [None] * 5
        # X is expected in NHWC layout (the Keras default)
        for i in range(5):
            blk = getattr(self, f'blk_{i}')
            cls_pred = getattr(self, f'cls_{i}')
            bbox_pred = getattr(self, f'bbox_{i}')
            X, anchors_list[i], cls_preds_list[i], bbox_preds_list[i] = \
                blk_forward(X, blk, sizes[i], ratios[i],
                            cls_pred, bbox_pred, training=training)
        anchors = tf.concat(anchors_list, axis=1)
        cls_preds = concat_preds(cls_preds_list)
        cls_preds = tf.reshape(cls_preds,
                               (tf.shape(cls_preds)[0], -1,
                                self.num_classes + 1))
        bbox_preds = concat_preds(bbox_preds_list)
        return anchors, cls_preds, bbox_preds

    def _compute_ssd_loss(self, cls_preds, cls_labels, bbox_preds,
                          bbox_labels, bbox_masks):
        num_classes = self.num_classes + 1
        cls = self.cls_loss(
            tf.reshape(cls_labels, [-1]),
            tf.reshape(cls_preds, [-1, num_classes]))
        bbox = tf.reduce_mean(
            tf.abs((bbox_preds * bbox_masks) -
                   (bbox_labels * bbox_masks)))
        return cls + bbox

    def train_step(self, data):
        features, target = data
        with tf.GradientTape() as tape:
            anchors, cls_preds, bbox_preds = self(features, training=True)
            bbox_labels, bbox_masks, cls_labels = d2l.multibox_target(
                anchors, target)
            loss = self._compute_ssd_loss(cls_preds, cls_labels,
                                          bbox_preds, bbox_labels, bbox_masks)
        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        # Accumulate metrics without syncing large tensors to host
        cls_correct = tf.reduce_sum(
            tf.cast(tf.argmax(cls_preds, axis=-1) ==
                    tf.cast(cls_labels, tf.int64), tf.int64))
        cls_total = tf.cast(tf.size(cls_labels), tf.int64)
        bbox_abs = tf.reduce_sum(
            tf.abs((bbox_preds * bbox_masks) - (bbox_labels * bbox_masks)))
        bbox_total = tf.cast(tf.size(bbox_labels), tf.float32)
        return {'loss': loss,
                'cls_correct': cls_correct, 'cls_total': cls_total,
                'bbox_abs': bbox_abs, 'bbox_total': bbox_total}

We create a model instance
and use it to perform forward propagation
on a minibatch of $256 \times 256$ images `X`.

As shown earlier in this section,
the first block outputs $32 \times 32$ feature maps.
Recall that
the second to fourth downsampling blocks
halve the height and width
and the fifth block uses global pooling.
Since 4 anchor boxes
are generated for each unit along spatial dimensions
of feature maps,
at all the five scales
a total of $(32^2 + 16^2 + 8^2 + 4^2 + 1)\times 4 = 5444$ anchor boxes are generated for each image.

In [ ]:
net = TinySSD(num_classes=1)
X = tf.zeros((32, 256, 256, 3))  # NHWC for Keras
anchors, cls_preds, bbox_preds = net(X)

print('output anchors:', anchors.shape)
print('output class preds:', cls_preds.shape)
print('output bbox preds:', bbox_preds.shape)

## Training

Now we will explain
how to train the single shot multibox detection model
for object detection.


### Reading the Dataset and Initializing the Model

To begin with,
let's read
the banana detection dataset
described in that section.

In [ ]:
batch_size = 32
train_iter, _ = d2l.load_data_bananas(batch_size)

There is only one class in the banana detection dataset. After defining the model,
we need to initialize its parameters and define
the optimization algorithm.

In [ ]:
net = TinySSD(num_classes=1)
net.compile(optimizer=keras.optimizers.SGD(learning_rate=0.2,
                                           weight_decay=5e-4))

### Defining Loss and Evaluation Functions

Object detection has two types of losses.
The first loss concerns classes of anchor boxes:
its computation
can simply reuse
the cross-entropy loss function
that we used for image classification.
The second loss
concerns offsets of positive (non-background) anchor boxes:
this is a regression problem.
For this regression problem,
however,
here we do not use the squared loss
described in that section.
Instead,
we use the $\ell_1$ norm loss,
the absolute value of the difference between
the prediction and the ground-truth.
The mask variable `bbox_masks` filters out
negative anchor boxes and illegal (padded)
anchor boxes in the loss calculation.
In the end, we sum up
the anchor box class loss
and the anchor box offset loss
to obtain the loss function for the model.

In [ ]:
# Loss functions are encapsulated in TinySSD._compute_ssd_loss and
# train_step; these module-level helpers mirror the other frameworks for
# use in evaluation after training.
_cls_loss = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')

def calc_loss(cls_preds, cls_labels, bbox_preds, bbox_labels, bbox_masks):
    batch_size = cls_preds.shape[0]
    num_classes = cls_preds.shape[2]
    # Per-example mean cross-entropy over anchors (shape: (batch_size,))
    cls = tf.reduce_mean(
        tf.reshape(_cls_loss(tf.reshape(cls_labels, [-1]),
                             tf.reshape(cls_preds, [-1, num_classes])),
                   [batch_size, -1]),
        axis=1)
    # Per-example mean L1 bbox loss (shape: (batch_size,)) to match PT/JAX
    bbox = tf.reduce_mean(
        tf.abs((bbox_preds - bbox_labels) * bbox_masks), axis=1)
    return cls + bbox

We can use accuracy to evaluate the classification results.
Due to the used $\ell_1$ norm loss for the offsets,
we use the *mean absolute error* to evaluate the
predicted bounding boxes.
These prediction results are obtained
from the generated anchor boxes and the
predicted offsets for them.

In [ ]:
def cls_eval(cls_preds, cls_labels):
    # Because the class prediction results are on the final dimension,
    # `argmax` needs to specify this dimension
    return float(tf.reduce_sum(
        tf.cast(tf.argmax(cls_preds, axis=-1) ==
                tf.cast(cls_labels, tf.int64), tf.int64)))

def bbox_eval(bbox_preds, bbox_labels, bbox_masks):
    return float(tf.reduce_sum(
        tf.abs((bbox_labels - bbox_preds) * bbox_masks)))

### Training the Model

When training the model,
we need to generate multiscale anchor boxes (`anchors`)
and predict their classes (`cls_preds`) and offsets (`bbox_preds`) in the forward propagation.
Then we label the classes (`cls_labels`) and offsets (`bbox_labels`) of such generated anchor boxes
based on the label information `Y`.
Finally, we calculate the loss function
using the predicted and labeled values
of the classes and offsets.
For concise implementations,
evaluation of the test dataset is omitted here.

In [ ]:
num_epochs, timer = 20, d2l.Timer()
animator = d2l.Animator(xlabel='epoch', xlim=[1, num_epochs],
                        legend=['class error', 'bbox mae'])
# Anchors depend only on the input *shape*, so for a fixed input size
# they're constant across the epoch. Grab them once here and reuse
# them in every batch's multibox_target call — saves one forward pass
# per step.
sample_features, _ = next(iter(train_iter))
sample_features = tf.cast(sample_features, tf.float32)
anchors_static, _, _ = net(sample_features, training=False)
# `multibox_target` mixes TF ops with NumPy / `.numpy()` calls, which
# can't run inside `@tf.function`. We keep it eager and only put the
# pure-TF gradient step under @tf.function. The wrapper closes over
# `net` so its trainable variables aren't @tf.function arguments
# (which would re-trace each call) — same pattern that worked for
# gan / dcgan.
@tf.function(reduce_retracing=True)
def _grad_step(features, bbox_labels, bbox_masks, cls_labels):
    with tf.GradientTape() as tape:
        _, cls_preds, bbox_preds = net(features, training=True)
        loss = net._compute_ssd_loss(cls_preds, cls_labels, bbox_preds,
                                     bbox_labels, bbox_masks)
    grads = tape.gradient(loss, net.trainable_variables)
    net.optimizer.apply_gradients(zip(grads, net.trainable_variables))
    cls_correct = tf.reduce_sum(
        tf.cast(tf.argmax(cls_preds, axis=-1) ==
                tf.cast(cls_labels, tf.int64), tf.int64))
    cls_total = tf.cast(tf.size(cls_labels), tf.int64)
    bbox_abs = tf.reduce_sum(
        tf.abs((bbox_preds * bbox_masks) - (bbox_labels * bbox_masks)))
    bbox_total = tf.cast(tf.size(bbox_labels), tf.float32)
    return loss, cls_correct, cls_total, bbox_abs, bbox_total
for epoch in range(num_epochs):
    # Accumulate metrics as TF tensors so the inner loop never forces
    # a host sync (the per-batch `int(logs['cls_correct'])` was a
    # significant bottleneck before).
    cls_correct_sum = tf.zeros((), dtype=tf.int64)
    cls_total = tf.zeros((), dtype=tf.int64)
    bbox_abs_total = tf.zeros((), dtype=tf.float32)
    bbox_total = tf.zeros((), dtype=tf.float32)
    for features, target in train_iter:
        timer.start()
        features = tf.cast(features, tf.float32)
        target = tf.cast(target, tf.float32)
        bbox_labels, bbox_masks, cls_labels = d2l.multibox_target(
            anchors_static, target)
        _, cc, ct, ba, bt = _grad_step(features, bbox_labels,
                                       bbox_masks, cls_labels)
        cls_correct_sum += cc
        cls_total += ct
        bbox_abs_total += ba
        bbox_total += bt
    cls_err = float(1 - cls_correct_sum / cls_total)
    bbox_mae = float(bbox_abs_total / bbox_total)
    animator.add(epoch + 1, (cls_err, bbox_mae))
print(f'class err {cls_err:.2e}, bbox mae {bbox_mae:.2e}')
print(f'{len(train_iter) * batch_size / timer.stop():.1f} examples/sec')

## Prediction

During prediction,
the goal is to detect all the objects of interest
on the image.
Below
we read and resize a test image,
converting it to
a four-dimensional tensor that is
required by convolutional layers.

In [ ]:
from PIL import Image as PILImage
img_pil = PILImage.open('../img/banana.jpg')
img = np.array(img_pil)
# img is (H, W, C); add batch dim for NHWC input
X = tf.expand_dims(tf.cast(img, tf.float32), axis=0)

Using the `multibox_detection` function below,
the predicted bounding boxes
are obtained
from the anchor boxes and their predicted offsets.
Then non-maximum suppression is used
to remove similar predicted bounding boxes.

In [ ]:
def predict(X):
    anchors, cls_preds, bbox_preds = net(X, training=False)
    # cls_preds: (batch, num_anchors, num_classes+1) -> softmax -> transpose
    # to (batch, num_classes+1, num_anchors) for multibox_detection
    cls_probs = tf.transpose(tf.nn.softmax(cls_preds, axis=-1), (0, 2, 1))
    output = d2l.multibox_detection(cls_probs, bbox_preds, anchors)
    # Drop padding rows (class index -1).
    mask = output[0, :, 0] != -1
    idx = tf.where(mask)[:, 0]
    return tf.gather(output[0], idx)

output = predict(X)

Finally, we display
all the predicted bounding boxes with
confidence 0.9 or above
as output.

In [ ]:
def display(img, output, threshold):
    d2l.set_figsize((5, 5))
    fig = d2l.plt.imshow(img)
    for row in output:
        score = float(row[1])
        if score < threshold:
            continue
        h, w = img.shape[:2]
        bbox = [row[2:6] * np.array((w, h, w, h), dtype=np.float32)]
        d2l.show_bboxes(fig.axes, bbox, '%.2f' % score, 'w')

display(img, output, threshold=0.9)

## Summary

* Single shot multibox detection is a multiscale object detection model. Via its base network and several multiscale feature map blocks, single-shot multibox detection generates a varying number of anchor boxes with different sizes, and detects varying-size objects by predicting classes and offsets of these anchor boxes (thus the bounding boxes).
* When training the single-shot multibox detection model, the loss function is calculated based on the predicted and labeled values of the anchor box classes and offsets.


## Exercises

1. Can you improve the single-shot multibox detection by improving the loss function? For example, replace $\ell_1$ norm loss with smooth $\ell_1$ norm loss for the predicted offsets. This loss function uses a square function around zero for smoothness, which is controlled by the hyperparameter $\sigma$:

$$
f(x) =
    \begin{cases}
    (\sigma x)^2/2,& \textrm{if }|x| < 1/\sigma^2\\
    |x|-0.5/\sigma^2,& \textrm{otherwise}
    \end{cases}
$$

When $\sigma$ is very large, this loss is similar to the $\ell_1$ norm loss. When its value is smaller, the loss function is smoother.

In [ ]:
def smooth_l1(data, scalar):
    cond = tf.abs(data) < 1 / (scalar ** 2)
    quad = (scalar * data) ** 2 / 2
    lin = tf.abs(data) - 0.5 / (scalar ** 2)
    return tf.where(cond, quad, lin)

sigmas = [10, 1, 0.5]
lines = ['-', '--', '-.']
x = tf.range(-2.0, 2.0, 0.1)
d2l.set_figsize()

for l, s in zip(lines, sigmas):
    y = smooth_l1(x, scalar=s)
    d2l.plt.plot(x, y, l, label='sigma=%.1f' % s)
d2l.plt.legend();

Besides, in the experiment we used cross-entropy loss for class prediction:
denoting by $p_j$ the predicted probability for the ground-truth class $j$, the cross-entropy loss is $-\log p_j$. We can also use the focal loss
[@Lin.Goyal.Girshick.ea.2017]: given hyperparameters $\gamma > 0$
and $\alpha > 0$, this loss is defined as:

$$ - \alpha (1-p_j)^{\gamma} \log p_j.$$

As we can see, increasing $\gamma$
can effectively reduce the relative loss
for well-classified examples (e.g., $p_j > 0.5$)
so the training
can focus more on those difficult examples that are misclassified.

In [ ]:
def focal_loss(gamma, x):
    return -(1 - x) ** gamma * tf.math.log(x)

x = tf.range(0.01, 1.0, 0.01)
for l, gamma in zip(lines, [0, 1, 5]):
    y = d2l.plt.plot(x, focal_loss(gamma, x), l, label='gamma=%.1f' % gamma)
d2l.plt.legend();

2. Due to space limitations, we have omitted some implementation details of the single shot multibox detection model in this section. Can you further improve the model in the following aspects:
    1. When an object is much smaller compared with the image, the model could resize the input image bigger.
    1. There are typically a vast number of negative anchor boxes. To make the class distribution more balanced, we could downsample negative anchor boxes.
    1. In the loss function, assign different weight hyperparameters to the class loss and the offset loss.
    1. Use other methods to evaluate the object detection model, such as those in the single shot multibox detection paper [@Liu.Anguelov.Erhan.ea.2016].

[Discussions](https://d2l.discourse.group/t/1604)